In [1]:
from dotenv import load_dotenv

load_dotenv()

True

#### LLM

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model='qwen3:4b', 
    num_predict=512,
)

#### define tools

In [3]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:  # 함수
    """숫자 a와 b를 더합니다."""
    return a+b

@tool
def multiply(a: int, b: int) -> int:  # 함수
    """숫자 a와 b를 곱합니다."""
    return a*b

In [4]:
# add(3, 4)  # 에러

- tool -> invoke 를 해줘야 함 (LLM 컴포넌트에 있는 메서드이기 때문에)

In [5]:
add.invoke({'a':3, 'b':4})  # 딕셔너리로 감싸서

7

#### LLM tools 활용

In [6]:
"""
# bind_functions 보다 bind_tools 를 사용하는게 더 추천되는 방식
"""

# 도구들을 리스트로 할당
llm_with_tools = llm.bind_tools([add, multiply])

In [7]:
# 테스트
query = '3 곱하기 5는?'

In [8]:
llm.invoke(query)  

# 도구를 호출하지 않고, 일반적으로 llm을 invoke 해서 활용하게 되면
# AIMessage 객체의 content 에 값이 담긴다. 

AIMessage(content='3 곱하기 5는 15입니다.  \n3 × 5 = 15  \n\n**해설**:  \n곱셈은', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-11T06:49:44.7072038Z', 'done': True, 'done_reason': 'length', 'total_duration': 60811206500, 'load_duration': 19495300, 'prompt_eval_count': 18, 'prompt_eval_duration': 395277000, 'eval_count': 512, 'eval_duration': 60366392000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a129b8-9073-76a1-9b1a-0b3ae13d344b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 512, 'total_tokens': 530})

In [9]:
llm_with_tools.invoke(query)

# contents 가 아닌 다른 키값에 tool_calls 가 생성됨

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-11T06:50:04.9005061Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20167764700, 'load_duration': 34936300, 'prompt_eval_count': 195, 'prompt_eval_duration': 105551000, 'eval_count': 215, 'eval_duration': 19955675000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a129b9-7e1a-73d3-b73d-1a81d4f71735-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'c41b2124-1b05-4714-9783-f0b153cca1f1', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 195, 'output_tokens': 215, 'total_tokens': 410})

In [10]:
response = llm_with_tools.invoke(query)

print("본문:", response.content)
print("도구 호출:", response.tool_calls) # 호출할 도구들이 리스트로 담긴다 (LLM이 필요한 도구를 쿼리에 맞게 직접 선택)
print("잘못된 도구 호출:", response.invalid_tool_calls)

본문: 
도구 호출: [{'name': 'multiply', 'args': {'a': 3, 'b': 5}, 'id': 'c738088c-1cc2-49b5-92e7-cb192aa29348', 'type': 'tool_call'}]
잘못된 도구 호출: []


#### LLM invoke tools
- LLM이 도구를 invoke 할 수 있게 해줘야 함

```text
messages: Sequence[AnyMessage]
                    │
                    ├─ SystemMessage → 모델의 역할·규칙·지침 (Developer)
                    ├─ HumanMessage  → 사용자의 질문·요청
                    ├─ AIMessage     → 모델의 답변 또는 도구 호출 요청
                    └─ ToolMessage   → 도구를 실제로 실행한 결과
```

- LLM이 ToolMessage 들을 호출할 수 있게 개발해야 한다
- LLM에서 히스토리 관리를 할때 메시지가 리스트 형태로 넘어가는 걸 활용

```
history = [
    ("human", "안녕하세요."),
    ("ai", "안녕하세요! 무엇을 도와드릴까요?"),
    ("human", "3 곱하기 5는 얼마인가요?"),
]
```
- 이렇게 넘겨서 invoke 를 해줘야 LLM이 도구를 사용하고 정상적으로 답변을 생성할 수 있음 (그냥 도구만 주면 정상적인 답변 생성X)

In [11]:
from langchain_core.messages import AnyMessage, HumanMessage
from typing import Sequence

# 사용자 쿼리 추가
human_message = HumanMessage(query)

# human_message 추가
message_list: Sequence[AnyMessage] = [human_message]

In [12]:
ai_message = llm_with_tools.invoke(message_list) # AIMessage 반환

In [13]:
ai_message.tool_calls  # 어떤 도구를 활용할지?

[{'name': 'multiply',
  'args': {'a': 3, 'b': 5},
  'id': 'cf48922f-d9e0-40a2-ae21-9775cd00503c',
  'type': 'tool_call'}]

In [14]:
message_list.append(ai_message)

In [18]:
# 도구들의 호출을 ai_message로 invoke
tool_message = multiply.invoke(ai_message.tool_calls[0])
tool_message

ToolMessage(content='15', name='multiply', tool_call_id='cf48922f-d9e0-40a2-ae21-9775cd00503c')

In [ ]:
message_list.append(tool_message)  # 도출된 ToolMessage를 추가

In [20]:
final_results = llm_with_tools.invoke(message_list)

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-11T06:53:41.8671688Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20970219000, 'load_duration': 22710000, 'prompt_eval_count': 243, 'prompt_eval_duration': 265887000, 'eval_count': 207, 'eval_duration': 20633938000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a129bc-ca7e-7df3-94f0-54f45b4dec96-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 243, 'output_tokens': 207, 'total_tokens': 450})

In [ ]:
""" final_results

AIMessage(content='3 곱하기 5는 15입니다.', additional_kwargs={}, response_metadata={'model': 'qwen3:4b', 'created_at': '2026-10-11T06:53:41.8671688Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20970219000, 'load_duration': 22710000, 'prompt_eval_count': 243, 'prompt_eval_duration': 265887000, 'eval_count': 207, 'eval_duration': 20633938000, 'logprobs': None, 'model_name': 'qwen3:4b', 'model_provider': 'ollama'}, id='lc_run--01a129bc-ca7e-7df3-94f0-54f45b4dec96-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 243, 'output_tokens': 207, 'total_tokens': 450})
"""